# Automated Vintage Matrix Generation from Real Excel Data

In a real-world scenario, you don't use randomly generated numbers; you have a massive Excel sheet of everything that happened up until today. 

This notebook demonstrates how to load a real `data/US/...xls` file into pandas, pair it with realistic publication schedules, and blindly strip out what a researcher in 2016 wouldn't have known.

In [1]:
import pandas as pd
from dfm_sp import VintageMaker, WeekdayRule, FixedDayRule

# 1. Load the real historical Excel file as a Pandas DataFrame
# Here we use the raw data from December 2016.
raw_data = pd.read_excel('data/US/2016-12-16.xls')

# Convert the string Date column into a proper DatetimeIndex
raw_data['Date'] = pd.to_datetime(raw_data['Date'])
raw_data.set_index('Date', inplace=True)

print("--- Original (Omniscient) Raw Excel Data (End of Matrix) ---")
display(raw_data[['PAYEMS', 'INDPRO', 'RSAFS', 'CPIAUCSL']].tail(4))

--- Original (Omniscient) Raw Excel Data (End of Matrix) ---


,PAYEMS,INDPRO,RSAFS,CPIAUCSL
Date,,,,
2016-09-01,144808.0,104.2429,462284.0,241.002
2016-10-01,144950.0,104.3185,465135.0,241.863
2016-11-01,145128.0,103.8589,465513.0,242.348
2016-12-01,NaN,NaN,NaN,NaN


### Defining Publication Rules for the Target APIs

We assign structural release patterns matching the SeriesIDs loaded in the DataFrame.

*   **`PAYEMS`** (Payrolls): Published 1st Friday of the following month.
*   **`INDPRO`** (Industrial Production): Published roughly on the 15th of the following month.
*   **`RSAFS`** (Retail Sales excluding Autos): Published ~15th of the following month.
*   **`CPIAUCSL`** (CPI Inflation): Published roughly the 12th of the following month.

In [ ]:
rules = {
    'PAYEMS': WeekdayRule(weekday=4, n=1, lag=1),
    'INDPRO': FixedDayRule(day=15, lag=1),
    'RSAFS': FixedDayRule(day=15, lag=1),
    'CPIAUCSL': FixedDayRule(day=12, lag=1)
}

maker = VintageMaker(rules_dict=rules)

### Triggering the Blinding Matrix

Let's see what the data looked like to an econometrician opening their computer on **December 10th, 2016**.

*   They *will* see `PAYEMS` for November (since the 1st Friday of December always happens before the 10th).
*   They *will not* see `INDPRO` or `RSAFS` or `CPIAUCSL` for November (publishing ~12th-15th). The Maker should strip these out dynamically as `NaN`.

In [3]:
simulation_target = "2016-12-10"

vintage_df = maker(raw_data, target_vintage_date=simulation_target)

print(f"--- Filtered Output Matrix (Exactly reflecting reality on {simulation_target}) ---")
display(vintage_df[['PAYEMS', 'INDPRO', 'RSAFS', 'CPIAUCSL']].tail(4))

# Notice that for November 2016 ('2016-11-01' row),
# 'PAYEMS' is kept intact, but the others are correctly scrubbed into NaNs.

--- Filtered Output Matrix (Exactly reflecting reality on 2016-12-10) ---


,PAYEMS,INDPRO,RSAFS,CPIAUCSL
Date,,,,
2016-09-01,144808.0,104.2429,462284.0,241.002
2016-10-01,144950.0,104.3185,465135.0,241.863
2016-11-01,145128.0,NaN,NaN,NaN
2016-12-01,NaN,NaN,NaN,NaN
